In [1]:
from googleapiclient.discovery import build
import pandas as pd
import tqdm
import isodate

from pytube.innertube import _default_clients
from pytube import cipher
import re
from langdetect import detect, DetectorFactory
# Set seed for reproducibility in langdetect
DetectorFactory.seed = 0
import langid

from googleapiclient.discovery import build
from datetime import datetime
import pandas as pd

_default_clients["ANDROID"]["context"]["client"]["clientVersion"] = "19.08.35"
_default_clients["IOS"]["context"]["client"]["clientVersion"] = "19.08.35"
_default_clients["ANDROID_EMBED"]["context"]["client"]["clientVersion"] = "19.08.35"
_default_clients["IOS_EMBED"]["context"]["client"]["clientVersion"] = "19.08.35"
_default_clients["IOS_MUSIC"]["context"]["client"]["clientVersion"] = "6.41"
_default_clients["ANDROID_MUSIC"] = _default_clients["ANDROID_CREATOR"]


def get_throttling_function_name(js: str) -> str:
    """Extract the name of the function that computes the throttling parameter.

    :param str js:
        The contents of the base.js asset file.
    :rtype: str
    :returns:
        The name of the function used to compute the throttling parameter.
    """
    function_patterns = [
        r'a\.[a-zA-Z]\s*&&\s*\([a-z]\s*=\s*a\.get\("n"\)\)\s*&&\s*'
        r'\([a-z]\s*=\s*([a-zA-Z0-9$]+)(\[\d+\])?\([a-z]\)',
        r'\([a-z]\s*=\s*([a-zA-Z0-9$]+)(\[\d+\])\([a-z]\)',
    ]
    #logger.debug('Finding throttling function name')
    for pattern in function_patterns:
        regex = re.compile(pattern)
        function_match = regex.search(js)
        if function_match:
            #logger.debug("finished regex search, matched: %s", pattern)
            if len(function_match.groups()) == 1:
                return function_match.group(1)
            idx = function_match.group(2)
            if idx:
                idx = idx.strip("[]")
                array = re.search(
                    r'var {nfunc}\s*=\s*(\[.+?\]);'.format(
                        nfunc=re.escape(function_match.group(1))),
                    js
                )
                if array:
                    array = array.group(1).strip("[]").split(",")
                    array = [x.strip() for x in array]
                    return array[int(idx)]

    raise RegexMatchError(
        caller="get_throttling_function_name", pattern="multiple"
    )

cipher.get_throttling_function_name = get_throttling_function_name

import os; api_key = os.getenv('YOUTUBE_API_KEY')


youtube = build('youtube', 'v3', developerKey=api_key )

In [ ]:
from googleapiclient.discovery import build

# api_key = "YOUR_API_KEY"
youtube = build("youtube", "v3", developerKey=api_key)

resp = youtube.search().list(
    q="@Fanta",
    type="channel",
    part="snippet",
    maxResults=1
).execute()

channel_id = resp["items"][0]["id"]["channelId"]

UCMcJfwaSdyMMn1u6BnoaKEw


'UCMcJfwaSdyMMn1u6BnoaKEw'

: 

# find ad pairs by channel

In [2]:
def get_channel_id(handle, api_key):
    youtube = build('youtube', 'v3', developerKey=api_key)
    request = youtube.search().list(
        part="snippet",
        q=handle,
        type="channel"
    )
    response = request.execute()
    
    for item in response['items']:
        if item['snippet']['channelTitle'].lower() == handle.lstrip('@').lower():
            return item['snippet']['channelId']
    return None

def get_playlists(channel_id, api_key):
    youtube = build('youtube', 'v3', developerKey=api_key)
    playlists = []
    request = youtube.playlists().list(
        part="id,snippet",
        channelId=channel_id,
        maxResults=50
    )
    response = request.execute()
    
    while request is not None:
        for item in response['items']:
            playlists.append({
                'id': item['id'],
                'title': item['snippet']['title']
            })
        request = youtube.playlists().list_next(request, response)
        if request:
            response = request.execute()
    
    return playlists

def get_videos_from_playlist(playlist_id, api_key):
    youtube = build('youtube', 'v3', developerKey=api_key)
    videos = []
    request = youtube.playlistItems().list(
        part="snippet,contentDetails",
        playlistId=playlist_id,
        maxResults=50
    )
    response = request.execute()
    
    while request is not None:
        for item in response['items']:
            video_id = item['contentDetails']['videoId']
            video_title = item['snippet']['title']
            videos.append({
                'id': video_id,
                'title': video_title
            })
        request = youtube.playlistItems().list_next(request, response)
        if request:
            response = request.execute()
    
    return videos

def get_video_details(video_id, api_key):
    youtube = build('youtube', 'v3', developerKey=api_key)
    request = youtube.videos().list(
        part="contentDetails,snippet",
        id=video_id
    )
    response = request.execute()
    if 'items' in response and len(response['items']) > 0:
        video = response['items'][0]
        duration = video['contentDetails']['duration']
        return {
            'id': video_id,
            'title': video['snippet']['title'],
            'duration': duration
        }
    else:
        return None

def parse_duration(duration):
    parsed_duration = isodate.parse_duration(duration)
    return parsed_duration.total_seconds()

In [3]:
def get_channel_language(channel_id, api_key):
    youtube = build('youtube', 'v3', developerKey=api_key)
    request = youtube.channels().list(
        part="snippet,brandingSettings",
        id=channel_id
    )
    response = request.execute()
    if 'items' in response and len(response['items']) > 0:
        channel = response['items'][0]
        if 'brandingSettings' in channel and 'channel' in channel['brandingSettings']:
            language = channel['brandingSettings']['channel'].get('defaultLanguage', 'en')
            return language
    return 'en'


In [4]:
handles  = ["@LEGO", "@vat19", "@IDJVideosTV", "@AngryBirds", "@dongtaypromotionofficial", "@Coca-Cola", "@Oddbods", "@MyLittlePonyOfficial", 
            "@MundoBitaVEVO", "@Disney", "@BMW", "@Google", "@GoPro", "@nike", "@redbull", "@VictoriasSecret", "@CHANEL", "@WWE", "@NintendoAmerica", 
            "@MonsterHigh", "@viralhog", "@DatPiff", "@Purple", "@headspace"]

In [5]:
@Fanta	UCMcJfwaSdyMMn1u6BnoaKEw

SyntaxError: invalid syntax (3989014102.py, line 1)

In [6]:
@motts	UCMfXv2enRXepxG92VoxfrEg

SyntaxError: invalid syntax (889965621.py, line 1)

In [7]:
handle = '@motts' #'@Coca-Cola'  # Replace with the actual handle
playlists = get_playlists('UCMfXv2enRXepxG92VoxfrEg', api_key) 

In [8]:
len(playlists)

24

In [9]:
all_videos = []
for playlist in tqdm.tqdm(playlists[0:]):
    videos = get_videos_from_playlist(playlist['id'], api_key)
    for video in videos:
        video_details = get_video_details(video['id'], api_key)
        if video_details:
            video_details['playlist_name'] = playlist['title']
            video_details['brand_name'] = handle 
            video_details['duration'] = parse_duration(video_details['duration'])
            all_videos.append(video_details)

100%|██████████| 24/24 [02:07<00:00,  5.32s/it]


In [13]:
df = pd.DataFrame(all_videos, columns=['brand_name', 'playlist_name', 'title', 'duration', 'id'])
df.rename(columns={'title': 'video_name', 'duration': 'video_duration', 'id': 'video_id'}, inplace=True)
filtered_df = df[df['video_duration'] < 70]
sorted_df = filtered_df.sort_values(by=['playlist_name', 'video_name', 'video_duration'])

sorted_df['url'] = 'https://www.youtube.com/watch?v='+sorted_df['video_id']
def make_hyperlink(url):
    return f'=HYPERLINK("{url}", "{url}")'

sorted_df['url'] = sorted_df['url'].apply(make_hyperlink)

excel_file_path = 'playlist/'+handle + '_filtered.xlsx'
sorted_df.to_excel(excel_file_path, index=False)
sorted_df

,brand_name,playlist_name,video_name,video_duration,video_id,url
268,@motts,GET READY FOR HALLOWEEN!,BEST HALLOWEEN PRANK FOR TRICK OR TREATERS! #s...,46.0,GbTtqbhmYwU,"=HYPERLINK(""https://www.youtube.com/watch?v=Gb..."
21,@motts,HAPPY 4TH OF JULY!,MOM HACKS | SAND CASTLE #shorts,56.0,HPkh6HNTCMw,"=HYPERLINK(""https://www.youtube.com/watch?v=HP..."
260,@motts,MOM HACKS,BEST HALLOWEEN PRANK FOR TRICK OR TREATERS! #s...,46.0,GbTtqbhmYwU,"=HYPERLINK(""https://www.youtube.com/watch?v=Gb..."
266,@motts,MOM HACKS,The RIGHT way to unpack soda into the fridge! ...,22.0,eGbeQj2SCD4,"=HYPERLINK(""https://www.youtube.com/watch?v=eG..."
8,@motts,VALENTINE'S DAY IDEAS!,HAPPY VALENTINE'S DAY! ;),49.0,-j8bKuNTDxg,"=HYPERLINK(""https://www.youtube.com/watch?v=-j..."


# find pairs by videos page

In [14]:
def get_videos(channel_id):
    videos = []
    next_page_token = None

    while True:
        request = youtube.search().list(
            part='id,snippet',
            channelId=channel_id,
            maxResults=50,
            type='video',
            pageToken=next_page_token
        )
        response = request.execute()

        for item in tqdm.tqdm(response['items']):
            video_id = item['id']['videoId']
            video_request = youtube.videos().list(
                part='snippet,contentDetails,statistics',
                id=video_id
            )
            video_response = video_request.execute()
            video_info = video_response['items'][0]

            video_data = {
                'id': video_id,
                'title': video_info['snippet']['title'],
                'description': video_info['snippet']['description'],
                'channelId': video_info['snippet']['channelId'],
                'channelTitle': video_info['snippet']['channelTitle'],
                'tags': ','.join(video_info['snippet'].get('tags', [])),
                'categoryId': video_info['snippet']['categoryId'],
                'liveBroadcastContent': video_info['snippet']['liveBroadcastContent'],
                'defaultLanguage': video_info['snippet'].get('defaultLanguage', ''),
                'localized_title': video_info['snippet']['localized']['title'],
                'localized_description': video_info['snippet']['localized']['description'],
                'duration': video_info['contentDetails']['duration'],
                'dimension': video_info['contentDetails']['dimension'],
                'definition': video_info['contentDetails']['definition'],
                'caption': video_info['contentDetails']['caption'],
                'licensedContent': video_info['contentDetails']['licensedContent'],
                'projection': video_info['contentDetails']['projection'],
                'viewCount': video_info['statistics'].get('viewCount', 0),
                'likeCount': video_info['statistics'].get('likeCount', 0),
                'dislikeCount': video_info['statistics'].get('dislikeCount', 0),
                'favoriteCount': video_info['statistics'].get('favoriteCount', 0),
                'commentCount': video_info['statistics'].get('commentCount', 0),
            }

            videos.append(video_data)

        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            break

    return videos

In [15]:
handles = pd.read_excel('handlestop50fastfood.xlsx')
handles.head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'handlestop50fastfood.xlsx'

In [16]:
for channel_id, handle  in zip(handles['channel'].values[35+8:], handles['handle'].values[35+8:]):
    videos = get_videos(channel_id)
    
    df = pd.DataFrame(videos)
    df['video_duration'] = df['duration'].apply(parse_duration)
    filtered_df = df[df['video_duration'] < 70]
    sorted_df = filtered_df.sort_values(by=['title', 'video_duration'])
    sorted_df['url'] = 'https://www.youtube.com/watch?v='+sorted_df['id']
    def make_hyperlink(url):
        return f'=HYPERLINK("{url}", "{url}")'
    
    sorted_df['url'] = sorted_df['url'].apply(make_hyperlink)
    sorted_df['handle'] = handle
    ordered_columns = ['handle', 'title', 'video_duration', 'id', 'url'] + [col for col in sorted_df.columns if col not in ['handle', 'title', 'video_duration', 'id', 'url']]
    sorted_df = sorted_df[ordered_columns]
    excel_file_path = 'videos/' + handle + '_filtered.xlsx'
    sorted_df.to_excel(excel_file_path, index=False)
    print('finish...', excel_file_path)

TypeError: list indices must be integers or slices, not str

In [ ]:
@motts	UCMfXv2enRXepxG92VoxfrEg

In [18]:
# channel_id = 'UC9tCymfWfzbe4vH-vskMkzg'
# handle = '@TelevisionCommercials'

handle = '@motts'
channel_id = 'UCMfXv2enRXepxG92VoxfrEg'

videos = get_videos(channel_id)

df = pd.DataFrame(videos)
df['video_duration'] = df['duration'].apply(parse_duration)
filtered_df = df[df['video_duration'] < 70]
sorted_df = filtered_df.sort_values(by=['title', 'video_duration'])
sorted_df['url'] = 'https://www.youtube.com/watch?v='+sorted_df['id']
def make_hyperlink(url):
    return f'=HYPERLINK("{url}", "{url}")'

sorted_df['url'] = sorted_df['url'].apply(make_hyperlink)
sorted_df['handle'] = handle
ordered_columns = ['handle', 'title', 'video_duration', 'id', 'url'] + [col for col in sorted_df.columns if col not in ['handle', 'title', 'video_duration', 'id', 'url']]
sorted_df = sorted_df[ordered_columns]
excel_file_path = 'videos/' + handle + '_filtered.xlsx'
sorted_df.to_excel(excel_file_path, index=False)
print('finish...', excel_file_path)

100%|██████████| 22/22 [00:01<00:00, 12.45it/s]

finish... videos/@motts_filtered.xlsx


# download by video id

In [20]:
#pairs = pd.read_excel('100pairs.xlsx', sheet_name='all')
pairs = pd.read_excel('100pairs.xlsx', sheet_name='beverage-en')
pairs.head(10)

,handle,title,video_duration,id,url,source
0,@Coca-Cola,Coca-Cola® Masterpiece,30,XFNqN0q_i2A,https://www.youtube.com/watch?v=XFNqN0q_i2A,NaN
1,@Coca-Cola,Coca-Cola® Masterpiece,15,Oo8MR5dyNlM,https://www.youtube.com/watch?v=Oo8MR5dyNlM,NaN
2,@Coca-Cola,A Recipe for Magic by Gigi Hadid | New Traditions,30,L3lI9FX5c6c,https://www.youtube.com/watch?v=L3lI9FX5c6c,NaN
3,@Coca-Cola,A Recipe for Magic with Gigi Hadid,15,w9Cq9OC8HD0,https://www.youtube.com/watch?v=w9Cq9OC8HD0,NaN
4,@Coca-Cola,Open That Coca-Cola 15s,15,CmO7IpHYkLk,https://www.youtube.com/watch?v=CmO7IpHYkLk,NaN
5,@Coca-Cola,"Open That Coca-Cola (Music by Tyler, The Creat...",30,Lysl59F2_Jo,https://www.youtube.com/watch?v=Lysl59F2_Jo,NaN
6,@Coca-Cola,One Coke® Away From Each Other - Real Magic,15,w3qm4Q3QZrw,https://www.youtube.com/watch?v=w3qm4Q3QZrw,NaN
7,@Coca-Cola,One Coke Away From Each Other - Real Magic,30,ijeLcMc_qcE,https://www.youtube.com/watch?v=ijeLcMc_qcE,NaN
8,@Coca-Cola,Chinese New Year Real Magic,15,OcCn_PMOoTE,https://www.youtube.com/watch?v=OcCn_PMOoTE,NaN
9,@Coca-Cola,Chinese New Year Real Magic,30,3jj4jzIsZiM,https://www.youtube.com/watch?v=3jj4jzIsZiM,NaN


In [ ]:
226/2

In [21]:
pairs['pair_id'] = ['b'+str(i) for i in range(23) for _ in range(2)]

In [34]:
pairs.head(10)

,handle,title,video_duration,id,url,source,pair_id
0,@Coca-Cola,Coca-Cola® Masterpiece,30,XFNqN0q_i2A,https://www.youtube.com/watch?v=XFNqN0q_i2A,NaN,b0
1,@Coca-Cola,Coca-Cola® Masterpiece,15,Oo8MR5dyNlM,https://www.youtube.com/watch?v=Oo8MR5dyNlM,NaN,b0
2,@Coca-Cola,A Recipe for Magic by Gigi Hadid | New Traditions,30,L3lI9FX5c6c,https://www.youtube.com/watch?v=L3lI9FX5c6c,NaN,b1
3,@Coca-Cola,A Recipe for Magic with Gigi Hadid,15,w9Cq9OC8HD0,https://www.youtube.com/watch?v=w9Cq9OC8HD0,NaN,b1
4,@Coca-Cola,Open That Coca-Cola 15s,15,CmO7IpHYkLk,https://www.youtube.com/watch?v=CmO7IpHYkLk,NaN,b2
5,@Coca-Cola,"Open That Coca-Cola (Music by Tyler, The Creat...",30,Lysl59F2_Jo,https://www.youtube.com/watch?v=Lysl59F2_Jo,NaN,b2
6,@Coca-Cola,One Coke® Away From Each Other - Real Magic,15,w3qm4Q3QZrw,https://www.youtube.com/watch?v=w3qm4Q3QZrw,NaN,b3
7,@Coca-Cola,One Coke Away From Each Other - Real Magic,30,ijeLcMc_qcE,https://www.youtube.com/watch?v=ijeLcMc_qcE,NaN,b3
8,@Coca-Cola,Chinese New Year Real Magic,15,OcCn_PMOoTE,https://www.youtube.com/watch?v=OcCn_PMOoTE,NaN,b4
9,@Coca-Cola,Chinese New Year Real Magic,30,3jj4jzIsZiM,https://www.youtube.com/watch?v=3jj4jzIsZiM,NaN,b4


In [2]:
import pandas as pd
import yt_dlp
import os

def download_video(row):
    ydl_opts = {
        'format': 'best',
        'outtmpl': f'../conference/videomedia/{row.pair_id}_{row.video_duration}_{row.id}.%(ext)s',
    }
    
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            ydl.download([row.url])
            print(f"Successfully downloaded: {row.id}")
        except Exception as e:
            print(f"Error downloading {row.id}: {str(e)}")

In [80]:
def download_video_fail(row):
    pair_id = row['pair_id']
    duration = row['video_duration']
    video_id = row['id']
    url = row['url']+video_id 
    ydl_opts = {
        'format': '137+140/best',
        'outtmpl': f'../conference/videomedia/{pair_id}_{duration}_{video_id}.%(ext)s',
    }
    
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            ydl.download(url)
            print(f"Successfully downloaded: {video_id}")
        except Exception as e:
            print(f"Error downloading {video_id}: {str(e)}")

In [10]:
row = {'pair_id':'b2', 'video_duration':30, 'id':'Lysl59F2_Jo','url':'https://www.youtube.com/watch?v='}
row = {'pair_id':'b10_1', 'video_duration':31, 'id':'6B14ME0zviQ','url':'https://www.youtube.com/watch?v='}

In [ ]:
15	7lf8IC0fmqM
30	jZlVEeiQlds

In [93]:
row = {'pair_id':'bb8', 'video_duration':15, 'id':'7lf8IC0fmqM','url':'https://www.youtube.com/watch?v='}

In [91]:
row = {'pair_id':'bb8', 'video_duration':30, 'id':'jZlVEeiQlds','url':'https://www.youtube.com/watch?v='}

In [94]:
download_video_fail(row)

[youtube] Extracting URL: https://www.youtube.com/watch?v=vk7UcK1nf_g
[youtube] vk7UcK1nf_g: Downloading webpage


[youtube] vk7UcK1nf_g: Downloading ios player API JSON
[youtube] vk7UcK1nf_g: Downloading player 28fd7348
[youtube] vk7UcK1nf_g: Downloading web player API JSON


[youtube] vk7UcK1nf_g: Downloading web player API JSON


[youtube] vk7UcK1nf_g: Downloading web player API JSON


[youtube] vk7UcK1nf_g: Downloading web player API JSON


[youtube] vk7UcK1nf_g: Downloading m3u8 information
[info] vk7UcK1nf_g: Downloading 1 format(s): 137+140
[download] Destination: ../conference/videomedia/bb7_16_vk7UcK1nf_g.f137.mp4
[download] 100% of    2.16MiB in 00:00:00 at 3.58MiB/s   
[download] Destination: ../conference/videomedia/bb7_16_vk7UcK1nf_g.f140.m4a
[download] 100% of  234.55KiB in 00:00:00 at 3.24MiB/s   
[Merger] Merging formats into "../conference/videomedia/bb7_16_vk7UcK1nf_g.mp4"
Deleting original file ../conference/videomedia/bb7_16_vk7UcK1nf_g.f137.mp4 (pass -k to keep)
Deleting original file ../conference/videomedia/bb7_16_vk7UcK1nf_g.f140.m4a (pass -k to keep)
Successfully downloaded: vk7UcK1nf_g


In [76]:
!yt-dlp -F https://www.youtube.com/watch?v=fX66MG6UUSI

[youtube] Extracting URL: https://www.youtube.com/watch?v=fX66MG6UUSI
[youtube] fX66MG6UUSI: Downloading webpage
[youtube] fX66MG6UUSI: Downloading ios player API JSON
[youtube] fX66MG6UUSI: Downloading player 28fd7348
[youtube] fX66MG6UUSI: Downloading web player API JSON
[youtube] fX66MG6UUSI: Downloading web player API JSON
[youtube] fX66MG6UUSI: Downloading web player API JSON
[youtube] fX66MG6UUSI: Downloading web player API JSON
[youtube] fX66MG6UUSI: Downloading m3u8 information
[info] Available formats for fX66MG6UUSI:
ID  EXT   RESOLUTION FPS CH │   FILESIZE  TBR PROTO │ VCODEC         VBR ACODEC      ABR ASR MORE INFO
─────────────────────────────────────────────────────────────────────────────────────────────────────────────
sb2 mhtml 48x27        7    │                 mhtml │ images                                 storyboard
sb1 mhtml 60x45        1    │                 mhtml │ images                                 storyboard
sb0 mhtml 120x90       1    │                 

In [ ]:
import os
import json
from typing import Optional, List, Dict
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

load_dotenv()

def build_yt(api_key: str):
    return build("youtube", "v3", developerKey=api_key)

def resolve_channel_id(api_key: str, seed: str) -> Optional[str]:
    """
    Accepts:
      - UC... channelId
      - @handle
      - https://www.youtube.com/@handle
      - https://www.youtube.com/channel/UC...
      - plain string -> fallback search (expensive)
    """
    yt = build_yt(api_key)
    s = seed.strip()

    # 1) Direct channelId
    if s.startswith("UC") and len(s) >= 20:
        return s

    # 2) URL forms
    if "youtube.com/channel/" in s:
        try:
            return s.split("youtube.com/channel/")[1].split("/")[0].split("?")[0]
        except Exception:
            pass

    # 3) Handle forms
    if "youtube.com/@" in s:
        s = "@" + s.split("youtube.com/@", 1)[1].split("/")[0].split("?")[0]
    if s.startswith("@"):
        handle = s[1:]
        resp = yt.channels().list(part="id", forHandle=handle).execute()
        items = resp.get("items", [])
        return items[0]["id"] if items else None

    # 4) Fallback: search (COSTLY: 100 units per call)
    resp = yt.search().list(part="snippet", q=s, type="channel", maxResults=1).execute()
    items = resp.get("items", [])
    if not items:
        return None
    return items[0]["id"]["channelId"]

def get_uploads_playlist_id(api_key: str, channel_id: str) -> Optional[str]:
    yt = build_yt(api_key)
    resp = yt.channels().list(part="contentDetails", id=channel_id).execute()
    items = resp.get("items", [])
    if not items:
        return None
    return items[0]["contentDetails"]["relatedPlaylists"]["uploads"]

def list_all_upload_video_ids(api_key: str, uploads_playlist_id: str) -> List[str]:
    yt = build_yt(api_key)
    ids: List[str] = []
    token = None

    while True:
        resp = yt.playlistItems().list(
            part="contentDetails",
            playlistId=uploads_playlist_id,
            maxResults=50,
            pageToken=token
        ).execute()

        for it in resp.get("items", []):
            ids.append(it["contentDetails"]["videoId"])

        token = resp.get("nextPageToken")
        if not token:
            break

    return ids

def fetch_video_details(api_key: str, video_ids: List[str]) -> List[Dict]:
    yt = build_yt(api_key)
    out: List[Dict] = []

    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        resp = yt.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(batch)
        ).execute()

        for v in resp.get("items", []):
            sn = v.get("snippet", {}) or {}
            cd = v.get("contentDetails", {}) or {}
            st = v.get("statistics", {}) or {}

            out.append({
                "id": v.get("id"),
                "title": sn.get("title", ""),
                "description": sn.get("description", ""),
                "publishedAt": sn.get("publishedAt", ""),
                "channelId": sn.get("channelId", ""),
                "channelTitle": sn.get("channelTitle", ""),
                "tags": sn.get("tags", []),
                "categoryId": sn.get("categoryId", ""),
                "defaultLanguage": sn.get("defaultLanguage", ""),
                "duration": cd.get("duration", ""),
                "definition": cd.get("definition", ""),
                "caption": cd.get("caption", ""),
                "viewCount": st.get("viewCount", 0),
                "likeCount": st.get("likeCount", 0),
                "commentCount": st.get("commentCount", 0),
            })

    return out

def export_channel_uploads(seed: str, out_json: str = "uploads_videos.json"):
    api_key = os.getenv("YOUTUBE_API_KEY")
    if not api_key:
        raise RuntimeError("Missing YOUTUBE_API_KEY in environment/.env")

    try:
        channel_id = resolve_channel_id(api_key, seed)
        if not channel_id:
            raise ValueError(f"Could not resolve channel from seed: {seed}")

        uploads_pid = get_uploads_playlist_id(api_key, channel_id)
        if not uploads_pid:
            raise ValueError(f"Could not get uploads playlist for channelId: {channel_id}")

        video_ids = list_all_upload_video_ids(api_key, uploads_pid)
        videos = fetch_video_details(api_key, video_ids)

        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "seed": seed,
                    "channelId": channel_id,
                    "uploadsPlaylistId": uploads_pid,
                    "videoCount": len(videos),
                    "videos": videos,
                },
                f,
                ensure_ascii=False,
                indent=2
            )

        print(f"Saved {len(videos)} videos to {out_json}")

    except HttpError as e:
        # show clean error
        print("YouTube API error:", e)

if __name__ == "__main__":
    # Examples:
    # export_channel_uploads("@heineken", "heineken_uploads.json")
    export_channel_uploads("@heineken", "heineken_uploads.json")
